In [ ]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()

In [ ]:
ngrok_ip = "0.tcp.in.ngrok.io:10122"

In [ ]:
catalog = dbutils.widgets.get("catalog")
checkpoints_dir = dbutils.widgets.get("checkpoints_dir")

In [ ]:
topics = {"sales": "sales_topic_new",
          "employees": "employees_topic_new",
          "expenses": "expenses_topic_new",
          "regions": "regions_topic_new"}

checkpoints = {"sales": f"abfss://checkpoints@jayveeradlsdevtest.dfs.core.windows.net/{checkpoints_dir}/brz_checkpoints/sales_checkpoint",
               "employees": f"abfss://checkpoints@jayveeradlsdevtest.dfs.core.windows.net/{checkpoints_dir}/brz_checkpoints/employees_checkpoint",
               "expenses": f"abfss://checkpoints@jayveeradlsdevtest.dfs.core.windows.net/{checkpoints_dir}/brz_checkpoints/expenses_checkpoint",
               "regions": f"abfss://checkpoints@jayveeradlsdevtest.dfs.core.windows.net/{checkpoints_dir}/brz_checkpoints/regions_checkpoint"
               }

In [ ]:
from pyspark.sql import functions as F

def consume_topic(topic_key, catalog):

      df = (spark.readStream
            .format("kafka")
            .option("kafka.bootstrap.servers", ngrok_ip)
            .option("subscribe", topics[topic_key])
            .option("startingOffsets", "earliest")
            .option("failOnDataLoss", "false")
            .load()
            )

      df = df.selectExpr("cast(key as string) as key",
                        "cast(value as string) as value",
                        "topic",
                        "partition",
                        "offset",
                        "timestamp").withColumn("processed_time", F.current_timestamp())

      query = (df.writeStream
            .format("delta")
            .option("checkpointLocation", checkpoints[topic_key])
            .trigger(availableNow = True)
            .outputMode("append")
            .table(f"{catalog}.brz.{topic_key}_raw")
            )

      return query

queries = {}

for topic_key, value in topics.items():

      q = consume_topic(topic_key, catalog)
      queries[topic_key] = q

for topic_key, q in queries.items():

      try:
            q.awaitTermination(60)
            print(f"Completed stream for {topic_key}")
      
      except Exception as e:

            raise(f"error in consuming {topic_key}: {e}")

In [ ]:
spark.sql(f"select * from {catalog}.brz.sales_raw limit 10")